# Fill And Adverse-Selection Research Model

This notebook is the fill-model layer of the market-making stack. It estimates whether a hypothetical passive quote is likely to be touched and whether that touch is expected to have positive or negative post-fill markout.

The current implementation uses bar-level touch proxies because we do not yet model queue position, order IDs, cancels, latency, or partial fills. Treat this notebook as a research bridge between fair value and a proper event-driven market-making simulator.

## Research Definition

Let `S_t` be the current reference price and let `d` be a proposed quote distance in basis points. For side `s`, define `s = +1` for a bid and `s = -1` for an ask. The hypothetical quote price is:

$$
Q_t(s,d) = S_t \exp\left(-s\frac{d}{10^4}\right)
$$

For a bid, a bar-level touch occurs if the future reference-price path trades down to the bid quote. For an ask, a touch occurs if the path trades up to the ask quote:

$$
\mathbb{1}^{\mathrm{touch}}_{t,s,d} =
\begin{cases}
\mathbb{1}\left[\min_{1 \le k \le H} S_{t+k} \le Q_t(+1,d)\right], & s=+1 \\
\mathbb{1}\left[\max_{1 \le k \le H} S_{t+k} \ge Q_t(-1,d)\right], & s=-1
\end{cases}
$$

Post-fill markout is measured from the maker's perspective after `M` minutes:

$$
m_{t,+1,d} = 10^4 \log\left(\frac{S_{t+M}}{Q_t(+1,d)}\right), \quad
m_{t,-1,d} = 10^4 \log\left(\frac{Q_t(-1,d)}{S_{t+M}}\right)
$$

The expected quote value before inventory penalty is approximated as:

$$
\widehat{V}_{t,s,d} = \widehat{p}^{\mathrm{touch}}_{t,s,d}\left(\widehat{m}_{t,s,d} - c_{\mathrm{maker}}\right)
$$

where `c_maker` is the maker fee or rebate in basis points. This is still not a production fill model because true fills depend on queue position and matching-engine events, not only future touches.

In [ ]:
from __future__ import annotations

import os
import sys
from datetime import datetime
from pathlib import Path

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

if "notebooks" not in sys.path:
    sys.path.append("notebooks")

from advanced_features import (
    FEATURE_DATASET_KEYS,
    available_dataset_dates,
    build_feature_set,
    build_targets,
    discover_datasets,
)

ROOT = Path(os.environ.get("MODL_WS_NORMALIZED_DIR", "/mnt/burner-archive/ws_normalized")).expanduser()
FEATURE_ROOT = Path(os.environ.get("MODL_WS_FEATURE_DIR", "/mnt/burner-archive/ws_features")).expanduser()
BAR_MINUTES = int(os.environ.get("MODL_BAR_MINUTES", "5"))
HORIZONS = tuple(int(value) for value in os.environ.get("MODL_HORIZONS", "5,15,30").split(","))
DATE_FILTER = [value.strip() for value in os.environ.get("MODL_FILL_DATES", "").split(",") if value.strip()]

INCLUDE_PARTIAL_DATES = True
QUOTE_DISTANCES_BPS = tuple(float(value) for value in os.environ.get("MODL_QUOTE_DISTANCES_BPS", "2,5,10,20").split(","))
TOUCH_HORIZON_MINUTES = int(os.environ.get("MODL_TOUCH_HORIZON_MINUTES", "15"))
MARKOUT_MINUTES = int(os.environ.get("MODL_MARKOUT_MINUTES", "30"))
MIN_FEATURE_OBS = int(os.environ.get("MODL_FILL_MIN_FEATURE_OBS", "50"))
MIN_TRAIN_OBS = int(os.environ.get("MODL_FILL_MIN_TRAIN_OBS", "250"))
MAKER_FEE_BPS = float(os.environ.get("MODL_MAKER_FEE_BPS", "0.0"))
LOGIT_C = float(os.environ.get("MODL_FILL_LOGIT_C", "0.25"))
RIDGE_ALPHA = float(os.environ.get("MODL_MARKOUT_RIDGE_ALPHA", "25.0"))
SAVE_OUTPUTS = False

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 260)
pd.set_option("display.max_colwidth", 220)

ROOT, BAR_MINUTES, QUOTE_DISTANCES_BPS, TOUCH_HORIZON_MINUTES, MARKOUT_MINUTES

## Date Inventory And Panel Build

Each date is built independently before labels are generated. This preserves point-in-time boundaries for rolling features and future touch labels. Partial dates are included and flagged.

In [ ]:
date_key_map = available_dataset_dates(ROOT)
date_inventory = pd.DataFrame(
    [
        {
            "date": date,
            "dataset_count": len(keys),
            "missing_required": sorted(set(FEATURE_DATASET_KEYS) - set(keys)),
            "is_full_dataset": set(FEATURE_DATASET_KEYS).issubset(keys),
        }
        for date, keys in sorted(date_key_map.items())
    ]
)
if DATE_FILTER:
    date_inventory = date_inventory[date_inventory["date"].isin(DATE_FILTER)].copy()
if not INCLUDE_PARTIAL_DATES:
    date_inventory = date_inventory[date_inventory["is_full_dataset"]].copy()

feature_by_date: dict[str, pd.DataFrame] = {}
model_by_date: dict[str, pd.DataFrame] = {}
build_rows: list[dict[str, object]] = []
build_errors: list[dict[str, object]] = []

for date in date_inventory["date"].tolist():
    date_tag = datetime.strptime(date, "%Y-%m-%d").strftime("%y-%m-%d")
    datasets = discover_datasets(ROOT, date_tag)
    try:
        feature_set = build_feature_set(datasets, horizons=HORIZONS, bar_minutes=BAR_MINUTES)
        features = feature_set.feature_matrix.copy()
        targets = build_targets(feature_set.reference_price, feature_set.term_structure, HORIZONS, bar_minutes=BAR_MINUTES)
        model = pd.concat([features, targets], axis=1).sort_index()
        feature_by_date[date] = features
        model_by_date[date] = model
        build_rows.append(
            {
                "date": date,
                "is_full_dataset": set(FEATURE_DATASET_KEYS).issubset(date_key_map[date]),
                "datasets": len(datasets),
                "rows": len(model),
                "feature_columns": features.shape[1],
                "reference_obs": int(model["reference_price"].notna().sum()),
            }
        )
    except Exception as exc:  # noqa: BLE001 - research notebook should report and continue
        build_errors.append({"date": date, "error_type": type(exc).__name__, "error": str(exc)})

if not model_by_date:
    raise RuntimeError("No fill-model dates built successfully")

build_summary = pd.DataFrame(build_rows).sort_values("date")
build_errors = pd.DataFrame(build_errors)
model_panel = pd.concat(model_by_date, names=["date", "minute"]).sort_index()

display(date_inventory)
display(build_summary)
if not build_errors.empty:
    display(build_errors)
model_panel.shape

## Quote Touch And Markout Labels

For each date, side, and quote distance, this section creates a long quote table. `touch` is a future path touch proxy. `markout_bps` is the maker-perspective markout after `MARKOUT_MINUTES`. `realized_quote_value_bps` is markout minus maker fee when touched and zero otherwise.

In [ ]:
touch_periods = max(1, int(round(TOUCH_HORIZON_MINUTES / BAR_MINUTES)))
markout_periods = max(1, int(round(MARKOUT_MINUTES / BAR_MINUTES)))


def future_min(series: pd.Series, periods: int) -> pd.Series:
    return series.shift(-1).iloc[::-1].rolling(periods, min_periods=periods).min().iloc[::-1]


def future_max(series: pd.Series, periods: int) -> pd.Series:
    return series.shift(-1).iloc[::-1].rolling(periods, min_periods=periods).max().iloc[::-1]


quote_label_frames: list[pd.DataFrame] = []
for date, model in model_by_date.items():
    reference = model["reference_price"].astype(float)
    fmin = future_min(reference, touch_periods)
    fmax = future_max(reference, touch_periods)
    mark_price = reference.shift(-markout_periods)
    base = pd.DataFrame(index=model.index)
    base["reference_price"] = reference
    base["future_min"] = fmin
    base["future_max"] = fmax
    base["mark_price"] = mark_price

    for side, side_sign in [("bid", 1.0), ("ask", -1.0)]:
        for distance_bps in QUOTE_DISTANCES_BPS:
            quote_price = reference * np.exp(-side_sign * distance_bps / 10_000.0)
            if side == "bid":
                touch = fmin <= quote_price
                markout_bps = 10_000.0 * np.log(mark_price / quote_price)
            else:
                touch = fmax >= quote_price
                markout_bps = 10_000.0 * np.log(quote_price / mark_price)
            frame = pd.DataFrame(index=pd.MultiIndex.from_product([[date], model.index], names=["date", "minute"]))
            frame["side"] = side
            frame["side_sign"] = side_sign
            frame["quote_distance_bps"] = distance_bps
            frame["quote_price"] = quote_price.to_numpy()
            frame["touch"] = touch.astype(float).to_numpy()
            frame["markout_bps"] = markout_bps.to_numpy()
            frame["realized_quote_value_bps"] = np.where(frame["touch"] == 1.0, frame["markout_bps"] - MAKER_FEE_BPS, 0.0)
            quote_label_frames.append(frame)

quote_labels = pd.concat(quote_label_frames).sort_index()
quote_labels = quote_labels.replace([np.inf, -np.inf], np.nan)
quote_summary_base = quote_labels.dropna(subset=["touch"]).copy()
quote_summary = (
    quote_summary_base.groupby(["side", "quote_distance_bps"], as_index=False)
    .agg(
        rows=("touch", "count"),
        touch_rate=("touch", "mean"),
        realized_quote_value_bps=("realized_quote_value_bps", "mean"),
    )
)
touched_summary = (
    quote_summary_base[quote_summary_base["touch"] == 1.0]
    .groupby(["side", "quote_distance_bps"], as_index=False)
    .agg(
        mean_markout_if_touched=("markout_bps", "mean"),
        adverse_touch_share=("markout_bps", lambda value: (value < 0).mean()),
    )
)
quote_summary = quote_summary.merge(touched_summary, on=["side", "quote_distance_bps"], how="left")
quote_summary = quote_summary[
    [
        "side",
        "quote_distance_bps",
        "rows",
        "touch_rate",
        "mean_markout_if_touched",
        "adverse_touch_share",
        "realized_quote_value_bps",
    ]
]
display(quote_summary)
quote_labels.tail(20)

## Feature Policy

Fill and markout models use state variables observable at quote time plus quote metadata. Raw price levels are excluded; quote distance and side carry the proposed quote geometry.

In [ ]:
ROLLING_SUFFIXES = (
    f"_diff_{BAR_MINUTES}m",
    "_mean_5m",
    "_z_5m",
    "_mean_15m",
    "_z_15m",
    "_mean_30m",
    "_z_30m",
)


def base_feature_name(feature: str) -> str:
    for suffix in ROLLING_SUFFIXES:
        if feature.endswith(suffix):
            return feature[: -len(suffix)]
    return feature


def feature_family(feature: str) -> str:
    base = base_feature_name(feature)
    rules = [
        ("hawkes_bsi", ("hawkes_bsi_",)),
        ("trade_flow", ("trade_flow_imbalance_",)),
        ("trade_activity", ("trade_volume_", "trade_trade_count_", "trade_vwap_")),
        ("book_microstructure", ("book_",)),
        ("cross_venue", ("cross_",)),
        ("spread_mid_arbitrage", ("mid_", "spread_")),
        ("realized_vol", ("rv_", "bpv_", "jump_")),
        ("returns", ("ret_",)),
        ("term_structure", ("iv_", "term_", "short_iv_decimal")),
        ("option_smile", ("smile_",)),
        ("futures_basis", ("basis_",)),
        ("funding", ("estimated_funding_rate",)),
    ]
    for family, prefixes in rules:
        if base.startswith(prefixes):
            return family
    return "other"


def is_disallowed_level(feature: str) -> bool:
    base = base_feature_name(feature)
    if base in {"reference_price", "log_price", "median_index_price", "mid_hibachi_minus_hyperliquid"}:
        return True
    if base.startswith(("book_mid_", "trade_vwap_")):
        return True
    if base.endswith(("_count", "_updates")) and not base.startswith("trade_trade_count_"):
        return True
    if base in {"active_options", "option_tick_count"}:
        return True
    return False


included_families = {
    "hawkes_bsi", "trade_flow", "trade_activity", "book_microstructure", "cross_venue",
    "spread_mid_arbitrage", "realized_vol", "returns", "term_structure", "option_smile",
    "futures_basis", "funding",
}
numeric_features = [column for column in model_panel.select_dtypes(include=[np.number]).columns if not column.startswith("target_")]
state_features = [
    column
    for column in numeric_features
    if feature_family(column) in included_families
    and not is_disallowed_level(column)
    and model_panel[column].replace([np.inf, -np.inf], np.nan).notna().sum() >= MIN_FEATURE_OBS
    and model_panel[column].nunique(dropna=True) > 1
]
quote_features = ["side_sign", "quote_distance_bps"]
model_features = quote_features + state_features

feature_catalog = pd.DataFrame(
    {
        "feature": state_features,
        "base_feature": [base_feature_name(column) for column in state_features],
        "family": [feature_family(column) for column in state_features],
        "is_rolling_transform": [base_feature_name(column) != column for column in state_features],
    }
)
display(feature_catalog.groupby("family").size().rename("feature_count").sort_values(ascending=False).to_frame())
len(model_features)

## Model Table

The model table joins quote labels with point-in-time state features. The same state row is repeated across quote sides and distances so the model can learn how quote geometry changes touch probability and markout.

In [ ]:
feature_rows = model_panel.reindex(columns=state_features)
quote_model_table = quote_labels.join(feature_rows, how="left")
quote_model_table = quote_model_table.dropna(subset=["touch", "markout_bps", "side_sign", "quote_distance_bps"])
quote_model_table["touched_markout_bps"] = np.where(quote_model_table["touch"] == 1.0, quote_model_table["markout_bps"], np.nan)

coverage = pd.DataFrame(
    {
        "non_null": quote_model_table[model_features + ["touch", "markout_bps"]].notna().sum(),
        "coverage": quote_model_table[model_features + ["touch", "markout_bps"]].notna().mean(),
    }
).sort_values("coverage")
display(coverage.head(30))
quote_model_table.shape

## Model Definitions

Touch probability is a regularized logistic model:

$$
\widehat{p}^{\mathrm{touch}} = \sigma(\beta_0 + z_t^\top\beta)
$$

Conditional markout is a ridge model fit only on touched quotes:

$$
\widehat{m} = \theta_0 + z_t^\top\theta
$$

The quote score is:

$$
\widehat{V} = \widehat{p}^{\mathrm{touch}}(\widehat{m} - c_{maker})
$$

In [ ]:
def make_touch_model() -> Pipeline:
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("logit", LogisticRegression(C=LOGIT_C, class_weight="balanced", max_iter=2_000)),
        ]
    )


def make_markout_model() -> Pipeline:
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=RIDGE_ALPHA)),
        ]
    )


def available_feature_columns(frame: pd.DataFrame, features: list[str], min_obs: int = MIN_FEATURE_OBS) -> list[str]:
    return [
        column
        for column in features
        if column in frame.columns
        and frame[column].replace([np.inf, -np.inf], np.nan).notna().sum() >= min_obs
        and frame[column].nunique(dropna=True) > 1
    ]


def touch_metrics(actual: pd.Series, predicted: pd.Series) -> dict[str, float]:
    pair = pd.concat([actual.rename("actual"), predicted.rename("predicted")], axis=1).dropna()
    if pair.empty:
        return {"n": 0, "touch_rate": np.nan, "brier": np.nan, "log_loss": np.nan, "auc": np.nan}
    has_two_classes = pair["actual"].nunique() == 2
    return {
        "n": int(len(pair)),
        "touch_rate": float(pair["actual"].mean()),
        "brier": float(brier_score_loss(pair["actual"], pair["predicted"])),
        "log_loss": float(log_loss(pair["actual"], pair["predicted"], labels=[0.0, 1.0])),
        "auc": float(roc_auc_score(pair["actual"], pair["predicted"])) if has_two_classes else np.nan,
    }


def markout_metrics(actual: pd.Series, predicted: pd.Series) -> dict[str, float]:
    pair = pd.concat([actual.rename("actual"), predicted.rename("predicted")], axis=1).dropna()
    if pair.empty:
        return {"markout_n": 0, "markout_rmse_bps": np.nan, "markout_mae_bps": np.nan, "markout_spearman": np.nan}
    err = pair["predicted"] - pair["actual"]
    return {
        "markout_n": int(len(pair)),
        "markout_rmse_bps": float(np.sqrt(np.mean(np.square(err)))),
        "markout_mae_bps": float(np.mean(np.abs(err))),
        "markout_spearman": float(pair["actual"].corr(pair["predicted"], method="spearman")) if pair["actual"].nunique() > 1 and pair["predicted"].nunique() > 1 else np.nan,
    }


## Walk-Forward Fill And Markout Models

For each date, models train only on prior dates. Dates without enough prior examples are skipped. This makes the current sample modest but keeps the diagnostic honest.

In [ ]:
walk_forward_frames: list[pd.DataFrame] = []
walk_forward_rows: list[dict[str, object]] = []
ordered_dates = sorted(model_by_date)

for test_date in ordered_dates:
    train_dates = [date for date in ordered_dates if date < test_date]
    test_frame = quote_model_table.xs(test_date, level="date", drop_level=False)
    if not train_dates:
        walk_forward_rows.append({"test_date": test_date, "status": "skipped_no_prior_dates", "train_obs": 0, "test_obs": len(test_frame)})
        continue

    train_frame = quote_model_table.loc[pd.IndexSlice[train_dates, :], :].copy()
    train_features = available_feature_columns(train_frame, model_features, min_obs=MIN_FEATURE_OBS)
    train_touch = train_frame.dropna(subset=["touch"])
    if len(train_touch) < MIN_TRAIN_OBS or train_touch["touch"].nunique() < 2 or not train_features:
        walk_forward_rows.append({"test_date": test_date, "status": "skipped_insufficient_train", "train_obs": len(train_touch), "test_obs": len(test_frame)})
        continue

    touch_model = make_touch_model()
    touch_model.fit(train_touch[train_features], train_touch["touch"])
    touch_prob = pd.Series(touch_model.predict_proba(test_frame.reindex(columns=train_features))[:, 1], index=test_frame.index, name="touch_prob")

    touched_train = train_frame[train_frame["touch"] == 1.0].dropna(subset=["markout_bps"])
    if len(touched_train) >= MIN_TRAIN_OBS // 4:
        markout_model = make_markout_model()
        markout_model.fit(touched_train[train_features], touched_train["markout_bps"])
        predicted_markout = pd.Series(markout_model.predict(test_frame.reindex(columns=train_features)), index=test_frame.index, name="predicted_markout_bps")
    else:
        predicted_markout = pd.Series(touched_train["markout_bps"].mean(), index=test_frame.index, name="predicted_markout_bps")

    predicted_value = touch_prob * (predicted_markout - MAKER_FEE_BPS)
    out = test_frame[["side", "side_sign", "quote_distance_bps", "touch", "markout_bps", "realized_quote_value_bps"]].copy()
    out["touch_prob"] = touch_prob
    out["predicted_markout_bps"] = predicted_markout
    out["predicted_quote_value_bps"] = predicted_value
    walk_forward_frames.append(out)

    row = {
        "test_date": test_date,
        "status": "fit",
        "train_dates": ",".join(train_dates),
        "train_obs": len(train_touch),
        "test_obs": len(test_frame),
        "feature_count": len(train_features),
        **touch_metrics(test_frame["touch"], touch_prob),
        **markout_metrics(test_frame.loc[test_frame["touch"] == 1.0, "markout_bps"], predicted_markout.loc[test_frame["touch"] == 1.0]),
    }
    walk_forward_rows.append(row)

walk_forward_summary = pd.DataFrame(walk_forward_rows)
walk_forward_quotes = pd.concat(walk_forward_frames).sort_index() if walk_forward_frames else pd.DataFrame(index=quote_model_table.index)
display(walk_forward_summary)
walk_forward_quotes.tail(20)

## All-Sample Research Fit

This section fits on all available data only to inspect feature effects and quote-distance behavior. It is not an out-of-sample performance estimate.

In [ ]:
research_features = available_feature_columns(quote_model_table, model_features, min_obs=MIN_FEATURE_OBS)
research_touch = quote_model_table.dropna(subset=["touch"])
research_touch_model = make_touch_model()
research_touch_model.fit(research_touch[research_features], research_touch["touch"])
research_touch_prob = pd.Series(research_touch_model.predict_proba(quote_model_table.reindex(columns=research_features))[:, 1], index=quote_model_table.index, name="research_touch_prob")

research_touched = quote_model_table[quote_model_table["touch"] == 1.0].dropna(subset=["markout_bps"])
research_markout_model = make_markout_model()
research_markout_model.fit(research_touched[research_features], research_touched["markout_bps"])
research_markout_pred = pd.Series(research_markout_model.predict(quote_model_table.reindex(columns=research_features)), index=quote_model_table.index, name="research_markout_bps")

research_quotes = quote_model_table[["side", "side_sign", "quote_distance_bps", "touch", "markout_bps", "realized_quote_value_bps"]].copy()
research_quotes["research_touch_prob"] = research_touch_prob
research_quotes["research_markout_bps"] = research_markout_pred
research_quotes["research_quote_value_bps"] = research_touch_prob * (research_markout_pred - MAKER_FEE_BPS)

research_metrics = pd.DataFrame(
    [
        {
            **touch_metrics(research_quotes["touch"], research_quotes["research_touch_prob"]),
            **markout_metrics(research_quotes.loc[research_quotes["touch"] == 1.0, "markout_bps"], research_quotes.loc[research_quotes["touch"] == 1.0, "research_markout_bps"]),
        }
    ]
)

touch_coef = pd.DataFrame({"feature": research_features, "touch_coef": research_touch_model.named_steps["logit"].coef_[0]})
touch_coef["abs_touch_coef"] = touch_coef["touch_coef"].abs()
touch_coef["family"] = touch_coef["feature"].map(feature_family)
markout_coef = pd.DataFrame({"feature": research_features, "markout_coef": research_markout_model.named_steps["ridge"].coef_})
markout_coef["abs_markout_coef"] = markout_coef["markout_coef"].abs()
markout_coef["family"] = markout_coef["feature"].map(feature_family)

display(research_metrics)
display(touch_coef.sort_values("abs_touch_coef", ascending=False).head(30))
display(markout_coef.sort_values("abs_markout_coef", ascending=False).head(30))

## Quote-Distance Ranking

These tables summarize expected quote value by side and distance. In production, this score must be combined with inventory, max order size, venue constraints, and risk limits before quoting.

In [ ]:
quote_ranking = (
    research_quotes.groupby(["side", "quote_distance_bps"], as_index=False)
    .agg(
        rows=("touch", "count"),
        realized_touch_rate=("touch", "mean"),
        predicted_touch_prob=("research_touch_prob", "mean"),
        predicted_markout_bps=("research_markout_bps", "mean"),
        realized_quote_value_bps=("realized_quote_value_bps", "mean"),
        predicted_quote_value_bps=("research_quote_value_bps", "mean"),
    )
)
quote_ranking_touched = (
    research_quotes[research_quotes["touch"] == 1.0]
    .groupby(["side", "quote_distance_bps"], as_index=False)
    .agg(mean_markout_if_touched=("markout_bps", "mean"))
)
quote_ranking = (
    quote_ranking.merge(quote_ranking_touched, on=["side", "quote_distance_bps"], how="left")
    .sort_values(["side", "predicted_quote_value_bps"], ascending=[True, False])
)

if not walk_forward_quotes.empty:
    walk_forward_ranking = (
        walk_forward_quotes.groupby(["side", "quote_distance_bps"], as_index=False)
        .agg(
            rows=("touch", "count"),
            realized_touch_rate=("touch", "mean"),
            predicted_touch_prob=("touch_prob", "mean"),
            realized_quote_value_bps=("realized_quote_value_bps", "mean"),
            predicted_quote_value_bps=("predicted_quote_value_bps", "mean"),
        )
        .sort_values(["side", "predicted_quote_value_bps"], ascending=[True, False])
    )
else:
    walk_forward_ranking = pd.DataFrame()

display(quote_ranking)
if not walk_forward_ranking.empty:
    display(walk_forward_ranking)

## Diagnostics

The first plot compares realized and predicted touch probability by quote distance. The second shows predicted quote value. The third shows walk-forward prediction quality when a fit exists.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
sns.lineplot(data=quote_ranking, x="quote_distance_bps", y="realized_touch_rate", hue="side", marker="o", ax=axes[0])
sns.lineplot(data=quote_ranking, x="quote_distance_bps", y="predicted_touch_prob", hue="side", marker="x", linestyle="--", ax=axes[0], legend=False)
axes[0].set_title("Touch Probability By Quote Distance")
axes[0].set_ylabel("probability")

sns.lineplot(data=quote_ranking, x="quote_distance_bps", y="predicted_quote_value_bps", hue="side", marker="o", ax=axes[1])
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("Predicted Quote Value, bps")
axes[1].set_ylabel("bps")
plt.tight_layout()

if not walk_forward_quotes.empty:
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)
    sns.scatterplot(data=walk_forward_quotes, x="touch_prob", y="touch", hue="side", alpha=0.6, ax=axes[0])
    axes[0].set_title("Walk-Forward Touch Predictions")
    touched = walk_forward_quotes[walk_forward_quotes["touch"] == 1.0]
    if not touched.empty:
        sns.scatterplot(data=touched, x="predicted_markout_bps", y="markout_bps", hue="side", alpha=0.6, ax=axes[1])
        axes[1].axline((0, 0), slope=1, color="black", linewidth=1)
        axes[1].set_title("Walk-Forward Markout Predictions On Touched Quotes")
    plt.tight_layout()


## Portfolio Manager Interpretation

This notebook estimates two questions needed before we quote:

1. How likely is a passive quote at distance `d` to be touched?
2. If touched, is the post-fill markout expected to compensate us for adverse selection and fees?

A quote candidate is attractive only when:

$$
\widehat{p}^{touch}(\widehat{m} - c_{maker}) - \mathrm{inventory\ penalty} - \mathrm{latency\ penalty} > 0
$$

The next research layer should combine this output with fair value and inventory to produce quote centers, skews, sizes, and cancel rules. A production implementation still needs event-level fills and queue-position simulation.

## Optional Save

Set `SAVE_OUTPUTS = True` to write quote labels, rankings, walk-forward diagnostics, and model coefficient tables.

In [ ]:
if SAVE_OUTPUTS:
    out_dir = FEATURE_ROOT / "fill_model" / f"bar_{BAR_MINUTES}m"
    out_dir.mkdir(parents=True, exist_ok=True)
    date_inventory.to_parquet(out_dir / "date_inventory.parquet")
    build_summary.to_parquet(out_dir / "build_summary.parquet")
    quote_labels.to_parquet(out_dir / "quote_labels.parquet")
    quote_summary.to_parquet(out_dir / "quote_summary.parquet")
    quote_ranking.to_parquet(out_dir / "quote_ranking.parquet")
    walk_forward_summary.to_parquet(out_dir / "walk_forward_summary.parquet")
    if not walk_forward_quotes.empty:
        walk_forward_quotes.to_parquet(out_dir / "walk_forward_quotes.parquet")
    research_metrics.to_parquet(out_dir / "research_metrics.parquet")
    touch_coef.to_parquet(out_dir / "touch_coefficients.parquet")
    markout_coef.to_parquet(out_dir / "markout_coefficients.parquet")
    print(f"wrote fill-model outputs to {out_dir}")
else:
    print("SAVE_OUTPUTS is false; nothing written")